In [1]:
import torch
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from qdrant_client.models import Distance, VectorParams

In [5]:
client = QdrantClient(url="http://localhost:6333")

In [4]:
client.create_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.DOT),
)

True

In [19]:
random_embeddings = np.random.normal(size=(10000, 1024))
random_embeddings /= np.sqrt((random_embeddings**2).sum(axis=-1))[:, None]

In [21]:
all_points = [PointStruct(id=l, vector = embed, payload={"city": "Berlin"}) for l, embed in enumerate(random_embeddings)]

In [22]:
operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=all_points[:1000],
)


In [27]:
query = np.random.normal(size=(1024,))
query = query/np.sqrt((query**2).sum())

In [28]:
search_result = client.query_points(
    collection_name="test_collection",
    query=query,
    with_payload=False,
    limit=3
).points

In [29]:
search_result

[ScoredPoint(id=845, version=2, score=0.10005581, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=791, version=2, score=0.08682226, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=810, version=2, score=0.086214915, payload=None, vector=None, shard_key=None, order_value=None)]

In [40]:
np.argmax(random_embeddings[:1000]@query)

np.int64(845)

## Trying my vector db

In [32]:
import pandas as pd
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer

In [12]:
chunks = load_dataset("parquet", data_files = "/proj/berzelius-2025-303/users/x_gabdu/random/ProjectAlbalat/albalat/data/processed/chunks.parquet")["train"]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [13]:
client = QdrantClient(url="http://localhost:6333")

In [14]:
test_query = np.random.normal(size=1024)

In [15]:
search_result = client.query_points(
    collection_name="v1_m_16_efConstruct_100",
    query=test_query,
    with_payload=False,
    limit=10
).points

print(search_result)

[ScoredPoint(id=3159698, version=2508, score=0.10004425, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=9913195, version=4845, score=0.09744263, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=12821862, version=14951, score=0.094055176, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4856663, version=2839, score=0.09112167, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=13251477, version=7992, score=0.08904648, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=14663692, version=9878, score=0.088321686, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=11759204, version=9539, score=0.08711243, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=12240681, version=7505, score=0.08561325, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1654163, version=9792, score=0.0

In [16]:
model_bf16 = SentenceTransformer("IEITYuan/Yuan-embedding-2.0-en", device="cpu", 
                            model_kwargs={"dtype": torch.bfloat16, "attn_implementation": "sdpa"})

In [49]:
def query_vector_index(query: str, top_k: int = 100):# -> Dataset:
    query_vector = model_bf16.encode(query, normalize_embeddings=True).astype(np.float16)
    search_result = client.query_points(
    collection_name="v1_m_16_efConstruct_100",
    query=query_vector,
    with_payload=False,
    limit=top_k
    ).points
    all_ids = [mapping_index_id[p.id] for p in search_result]
    all_id_scores = {mapping_index_id[p.id]: p.score for p in search_result}
    relevant_results = chunks.select(all_ids)
    #sorted_p = sorted([(all_id_scores[mapping_index_id[p["index"]]], p["index"], p["text_ids"], p["paragraphs"]) for p in relevant_results], reverse=True)
    sorted_p = [(all_id_scores[mapping_index_id[p["index"]]], p["index"], p["text_ids"], p["paragraphs"]) for p in relevant_results]
    df = pd.DataFrame.from_records(sorted_p)
    df.columns = ["scores", "ids", "text_ids", "paragraphs"]
    return df

def apply_cutoff_scores(df_results: pd.DataFrame, threshold: float) -> pd.DataFrame:
    return df_results[df_results.scores > threshold]


def query_endpoint(query: str, threshold: float, top_k: int = 100) -> pd.DataFrame:
    df_results = query_vector_index(query, top_k)
    df_threshold = apply_cutoff_scores(df_results, threshold)
    return df_threshold
    

In [18]:
from tqdm import tqdm
#mapping_index_id = {p["index"]:i for i, p in tqdm(enumerate(chunks))}

15538561it [06:50, 37859.55it/s]


In [37]:
query_text = "It was a evening of July. The sun slowly went down while the sky adorned itself with the most beautiful pink colors."
query_text2 = """It was about seven o'clock of an evening in late summer, and across that bleak, barren bit of land the sun was just setting. 
                As they drove along, it sparkled on the window panes of the houses and lit up the cross on the Catholic church; beyond the village 
                it seemed to confine itself to the rocks by the wayside. It turned them a dull soft gold. A strong salt breeze was blowing."""

query_text3 = "The storm was powerful and the waves humonguous. The ship threatened to collapse at any moment."
query_text4 = "I was standing here, among the dead bodies. The morgue felt cold, and I was cold inside, looking at my father lying on a table."
query_text5 = "The plane glided in the air. Its two gigantic wings were reflecting the sunlight while its engine was roaring in the sky."

In [47]:
df_results = query_vector_index(query_text5, 1000)
df_thresholded = apply_cutoff_scores(df_results, 0.6)

In [48]:
df_thresholded

,scores,ids,text_ids,paragraphs
0,0.855526,9560144,50622,"The big plane circled in the air, rising stead..."
1,0.815334,14470846,71441,"Across the patch a shape moved, glinting metal..."
2,0.802166,9560094,50622,The dark wings of the plane stretched out from...
3,0.794746,4801531,33481,"The sun was now in the zenith. The clouds, hav..."
4,0.790512,7235779,41586,"It sped onward, heavily, its almost transparen..."
...,...,...,...,...
995,0.723927,8161668,44782,"On came the plane, losing altitude with every ..."
996,0.723915,11715005,59066,"This evening, at sunset, Dick and I drove past..."
997,0.723892,8384489,45721,"As the day wore on, the gray of the sky paled ..."
998,0.723850,5341264,35180,He turned away from it and stood staring upwar...


In [50]:
eval_dataset = [query_text, query_text2, query_text3, query_text4, query_text5]

In [3]:
import os
import ragas
import asyncio
from openai import AsyncOpenAI
from ragas_examples.improve_rag.rag import RAG, BM25Retriever

ModuleNotFoundError: No module named 'ragas_examples'

In [55]:
os.environ["OPENAI_API_KEY"] = openai_api_key

In [60]:
openai_client = AsyncOpenAI()

# Create retriever and RAG system

# Query the system
question = "What architecture is the `tokenizers-linux-x64-musl` binary designed for?"
result = query_endpoint(question, 0.8)
print(f"Answer: {result['answer']}")

KeyError: 'answer'